This notebook implements our final CIFAR-10 image classification model, which achieved **88.80% accuracy on the Kaggle public leaderboard**.

The network is a custom preactivation Wide ResNet-28 with **4,923,705 trainable parameters**, satisfying the competition's five-million-parameter limit.

The model combines:
- **AutoAugment:** Learns from strongly augmented image variations.
- **MixUp and CutMix:** Regularize training by combining images and labels.
- **Sharpness-Aware Minimization (SAM):** Encourages solutions less sensitive to small weight perturbations.
- **Exponential Moving Average (EMA):** Maintains a smoothed copy of the model weights.

The notebook supports three execution modes: `validate`, `full`, and `predict`. Together, they support model development, final training, and competition submission generation.

**Dataset:** 50,000 labeled training images and 2,000 unlabeled competition test images, each of size 32 × 32 × 3.

**Final evaluation:** 96.98% local validation accuracy for the development checkpoint; 88.80% Kaggle public leaderboard accuracy for the separate full-data checkpoint.


In [ ]:
import os, json, time, random, math, csv, copy, warnings
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms
from torchvision.datasets import CIFAR10, ImageFolder
from sklearn.model_selection import train_test_split
from datetime import datetime, timezone

from google.colab import drive
drive.mount('/content/drive')
DRIVE_PROJECT = Path('/content/drive/MyDrive/CIFAR10_Competition')
DRIVE_PROJECT.mkdir(parents=True, exist_ok=True)

# ============================================================
# EXPERIMENT CONFIGURATION AND EXECUTION MODES
# ============================================================
# This cell imports dependencies, mounts Google Drive, defines
# dataset/checkpoint paths, configures training hyperparameters,
# and establishes reproducibility through a fixed random seed.
#
# Three execution modes are supported:
#
# 1. MODE = 'validate'
#    Splits 50,000 labeled images into 45,000 training and
#    5,000 validation images using stratified sampling.
#    Trains the model and compares raw and EMA validation
#    accuracy to select the best checkpoint.
#
# 2. MODE = 'full'
#    Trains the selected architecture from scratch using all
#    50,000 labeled images for 240 epochs.
#    No validation data is held out in this mode.
#    Saves the final EMA checkpoint for competition inference.
#
# 3. MODE = 'predict'
#    Skips training, loads an existing full-data checkpoint,
#    predicts labels for the 2,000 competition test images,
#    and generates a Kaggle-compatible submission CSV.
#
# The submitted configuration uses MODE = 'predict'.
# The pretrained checkpoint here refers only to our own
# trained checkpoint; no external pretrained model is used.
# ============================================================

# Select one execution stage:
# 'validate' = train on 45k images and evaluate on 5k labels
# 'full'     = retrain from scratch on all 50k labeled images
# 'predict'  = load final checkpoint and predict unlabeled test data
# For submission reproduction, leave this set to 'predict'.

MODE = 'predict'  # 'validate', 'full', or 'predict'
SEED = 2026
DATA_ROOT = Path('/content/cifar_data')
DATA_ZIP = DRIVE_PROJECT / 'data.zip'
EXTRACT_ARCHIVE = True
TRAIN_DIR = '/content/cifar_data/train_images'
TEST_DIR = '/content/cifar_data/test_images'
TRAIN_CSV = '/content/cifar_data/train_labels.csv'

# Path to the final full-data model checkpoint.
# Prediction mode loads this checkpoint instead of retraining.
# The notebook verifies that its architecture and training
# configuration match the expected final WRN-28 experiment.

CHECKPOINT_PATH = '/content/drive/MyDrive/CIFAR10_Competition/runs/full_wrn28_p4923705_sam0p05_aa_mixup_cutmix_p65_last30_ema0p999_seed2026_best.pt'
SAMPLE_CSV = '/content/cifar_data/sample_submission.csv'
OUTPUT = DRIVE_PROJECT / 'runs'
OUTPUT.mkdir(parents=True, exist_ok=True)


# Core optimization settings:
# 240 training epochs, batch size 128, SGD learning rate 0.1,
# momentum 0.9, and L2 weight decay of 0.0005.

EPOCHS=240; BATCH_SIZE=128; LR=0.1; WEIGHT_DECAY=5e-4; MOMENTUM=0.9

# SAM_RHO controls the magnitude of the SAM weight perturbation.
# EMA_DECAY controls smoothing of the moving-average weights.
# EMA tracking begins at epoch 10.
# The learning rate warms up during the first five epochs.

SAM_RHO=0.05; EMA_DECAY=0.999; EMA_START_EPOCH=10; WARMUP_EPOCHS=5

# During the first 210 epochs, each batch has a 65% chance
# of using sample mixing. MixUp and CutMix are selected with
# equal conditional probability.
# Mixing is disabled for the last 30 epochs.

MIX_PROB=0.65
MIXUP_ALPHA=0.2
CUTMIX_ALPHA=1.0
MIXUP_FRACTION=0.5
FINAL_UNMIXED_EPOCHS=30
AUTOAUGMENT_POLICY = transforms.AutoAugmentPolicy.CIFAR10
NUM_WORKERS=min(4, os.cpu_count() or 2); LABEL_SMOOTHING=0.05
CUTOUT_SIZE=0

# Enable horizontal-flip test-time augmentation.
# Predictions from the original and flipped images are
# averaged before selecting the final class.

USE_TTA=True
RESUME=True
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
seed_everything(SEED)
assert MODE in ('validate','full','predict')
assert 0 <= MIX_PROB <= 1 and 0 <= MIXUP_FRACTION <= 1 and FINAL_UNMIXED_EPOCHS < EPOCHS
assert SAM_RHO>=0 and 0 <= LABEL_SMOOTHING < 1 and 0 < EMA_DECAY < 1
print('PyTorch:',torch.__version__,'device:',DEVICE,'GPU:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')
print('DATA_ZIP:', DATA_ZIP, 'exists:', DATA_ZIP.is_file())
print('DATA_ROOT:',DATA_ROOT.resolve(), 'OUTPUT:', OUTPUT.resolve())
if not DATA_ZIP.is_file():
    raise FileNotFoundError(f'Upload data.zip to Google Drive folder {DRIVE_PROJECT} before proceeding')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PyTorch: 2.11.0+cu130 device: cuda GPU: NVIDIA L4
DATA_ZIP: /content/drive/MyDrive/CIFAR10_Competition/data.zip exists: True
DATA_ROOT: /content/cifar_data OUTPUT: /content/drive/MyDrive/CIFAR10_Competition/runs


## 1. Dataset Preparation and Archive Verification

The competition dataset is stored as `data.zip` in Google Drive. This section extracts it into temporary local storage for faster image loading during training and inference.

A SHA-256 fingerprint identifies the archive. If the archive is unchanged and has already been extracted, redundant extraction is skipped.

Before extraction, the notebook checks archive paths to prevent unsafe file traversal and rejects symbolic-link entries. It then prints the extracted file structure and previews CSV metadata to verify that the expected dataset files are present.


In [ ]:
import zipfile, hashlib, shutil

# Compute a SHA-256 fingerprint of the dataset archive.
# This identifies whether the ZIP file has changed since
# the most recent extraction.

def archive_fingerprint(path, block=8*1024*1024):
    h=hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda: f.read(block), b''):
            h.update(chunk)
    return h.hexdigest()

stamp = DATA_ROOT / '.archive_sha256'
fingerprint = archive_fingerprint(DATA_ZIP)
if EXTRACT_ARCHIVE and (not stamp.exists() or stamp.read_text().strip()!=fingerprint):
    if DATA_ROOT.exists():
        shutil.rmtree(DATA_ROOT)
    DATA_ROOT.mkdir(parents=True,exist_ok=True)
    with zipfile.ZipFile(DATA_ZIP) as z:
        root=DATA_ROOT.resolve()
        for info in z.infolist():
            dest=(root/info.filename).resolve()
            if not dest.is_relative_to(root):
                raise ValueError(f'Unsafe archive path: {info.filename}')
            if ((info.external_attr >> 16) & 0o170000) == 0o120000:
                raise ValueError(f'Symlink entry not allowed: {info.filename}')
        z.extractall(root)
    stamp.write_text(fingerprint)
    print('Extracted Drive ZIP to temporary local storage:',DATA_ROOT)
elif EXTRACT_ARCHIVE:
    print('Already extracted this ZIP in the current runtime; skipping.')
if not DATA_ROOT.exists():
    raise FileNotFoundError(f'Data root not found: {DATA_ROOT}. Attach the competition dataset or specify DATA_ROOT.')
files = sorted(p for p in DATA_ROOT.rglob('*') if p.is_file())
print('File count:',len(files))
print('Extensions:',dict(Counter(p.suffix.lower() for p in files)))
for p in files[:80]: print(p.relative_to(DATA_ROOT))
print('CSV files:')
for p in [p for p in files if p.suffix.lower()=='.csv'][:15]:
    try: print(str(p), pd.read_csv(p,nrows=3).to_dict(orient='list'))
    except Exception as e: print('Unable to preview',p,repr(e))
print('Sample submissions:',[str(p) for p in files if 'sample' in p.name.lower() and p.suffix.lower()=='.csv'])

Already extracted this ZIP in the current runtime; skipping.
File count: 52004
Extensions: {'': 1, '.csv': 3, '.png': 52000}
.archive_sha256
sample_submission.csv
test.csv
test_images/test_0000.png
test_images/test_0001.png
test_images/test_0002.png
test_images/test_0003.png
test_images/test_0004.png
test_images/test_0005.png
test_images/test_0006.png
test_images/test_0007.png
test_images/test_0008.png
test_images/test_0009.png
test_images/test_0010.png
test_images/test_0011.png
test_images/test_0012.png
test_images/test_0013.png
test_images/test_0014.png
test_images/test_0015.png
test_images/test_0016.png
test_images/test_0017.png
test_images/test_0018.png
test_images/test_0019.png
test_images/test_0020.png
test_images/test_0021.png
test_images/test_0022.png
test_images/test_0023.png
test_images/test_0024.png
test_images/test_0025.png
test_images/test_0026.png
test_images/test_0027.png
test_images/test_0028.png
test_images/test_0029.png
test_images/test_0030.png
test_images/test_0031.

## 2. Custom Wide ResNet-28 Architecture

We use a **preactivation Wide Residual Network with depth 28**. The network has three residual groups, each containing four residual blocks. Each block uses two 3 × 3 convolutions with Batch Normalization and ReLU.

### Network structure

| Component | Output channels | Spatial resolution |
|---|---:|---:|
| Initial convolution | 16 | 32 × 32 |
| Residual group 1 (4 blocks) | 59 | 32 × 32 |
| Residual group 2 (4 blocks) | 117 | 16 × 16 |
| Residual group 3 (4 blocks) | 235 | 8 × 8 |
| BatchNorm + ReLU | 235 | 8 × 8 |
| Global average pooling | 235 | 1 × 1 |
| Fully connected classifier | 10 | Class logits |

**Why a Wide ResNet?**

Residual shortcuts improve gradient propagation through deeper convolutional networks. Widening the residual groups increases representation capacity, while the custom channel counts `(16, 59, 117, 235)` allow us to use most of the permitted parameter budget without exceeding five million parameters.

The depth follows the Wide ResNet relation:

`depth = 6n + 4`

With `n = 4`, the network has 28 layers under this counting convention.

Projection shortcuts use 1 × 1 convolutions when the channel count or spatial resolution changes.

**Parameter budget:** 4,923,705 trainable parameters.


In [ ]:
# Define one preactivation residual block.
# BatchNorm and ReLU are applied before convolution.
# Two 3x3 convolutions learn residual features, while the
# shortcut preserves information from the input.
# A 1x1 projection is used when dimensions change.

class WideBlock(nn.Module):
    def __init__(self, cin, cout, stride, dropout=0.):
        super().__init__()
        self.bn1=nn.BatchNorm2d(cin)
        self.conv1=nn.Conv2d(cin,cout,3,stride,1,bias=False)
        self.bn2=nn.BatchNorm2d(cout)
        self.conv2=nn.Conv2d(cout,cout,3,1,1,bias=False)
        self.drop=nn.Dropout(p=dropout)
        self.shortcut=nn.Identity() if cin==cout and stride==1 else nn.Conv2d(cin,cout,1,stride,bias=False)
    def forward(self,x):
        z=F.relu(self.bn1(x),inplace=False)
        y=self.conv1(z)
        y=self.conv2(self.drop(F.relu(self.bn2(y),inplace=True)))
        return y+self.shortcut(x)

# Assemble the complete WRN-28.
# The network contains:
# - one 3x3 convolutional stem
# - three residual groups, with four blocks per group
# - BatchNorm and ReLU after the residual groups
# - global average pooling and a 10-class linear head
# Downsampling occurs at the first block of groups 2 and 3.

class WideResNet(nn.Module):
    def __init__(self,depth=28,channels=(16,59,117,235),nclasses=10,dropout=0.):
        super().__init__()
        assert (depth-4)%6==0
        n=(depth-4)//6; assert len(channels)==4 and channels[0]==16
        self.stem=nn.Conv2d(3,16,3,1,1,bias=False)
        in_c=16; stages=[]
        for idx,out_c in enumerate(channels[1:]):
            blocks=[]
            for b in range(n):
                stride=2 if idx>0 and b==0 else 1
                blocks.append(WideBlock(in_c,out_c,stride,dropout)); in_c=out_c
            stages.append(nn.Sequential(*blocks))
        self.stages=nn.Sequential(*stages)
        self.bn=nn.BatchNorm2d(in_c)
        self.fc=nn.Linear(in_c,nclasses)
        for m in self.modules():
            if isinstance(m,nn.Conv2d): nn.init.kaiming_normal_(m.weight,mode='fan_out',nonlinearity='relu')
            elif isinstance(m,nn.BatchNorm2d): nn.init.ones_(m.weight); nn.init.zeros_(m.bias)
    def forward(self,x):
        x=self.stem(x); x=self.stages(x)
        x=F.relu(self.bn(x),inplace=True)
        return self.fc(F.adaptive_avg_pool2d(x,1).flatten(1))

model=WideResNet().to(DEVICE)
# Count all trainable parameters and enforce the competition's
# strict maximum of five million parameters.
# Expected count: 4,923,705.

PARAMS=sum(p.numel() for p in model.parameters() if p.requires_grad)
assert PARAMS == 4_923_705 and PARAMS <= 5_000_000, f'Invalid parameter budget: {PARAMS:,}'
print('Verified trainable params:',f'{PARAMS:,}')
with torch.no_grad():
    output=model(torch.zeros(2,3,32,32,device=DEVICE))
    assert tuple(output.shape)==(2,10)
print('Output shape:',tuple(output.shape))

model.train()
# Verify that backward propagation produces finite gradients
# for all trainable parameters before attempting full training.
smoke_x=torch.randn(2,3,32,32,device=DEVICE)
smoke_loss=F.cross_entropy(model(smoke_x),torch.tensor([1,2],device=DEVICE))
smoke_loss.backward()
assert torch.isfinite(smoke_loss).item()
assert all(p.grad is not None and torch.isfinite(p.grad).all().item() for p in model.parameters() if p.requires_grad), 'Missing/non-finite gradients'
model.zero_grad(set_to_none=True)
model.eval()
with torch.no_grad():
    a=model(torch.zeros(2,3,32,32,device=DEVICE))
    b=model(torch.zeros(2,3,32,32,device=DEVICE))
    assert torch.equal(a,b), 'Inference is not deterministic'
print('PASS: architecture forward, backward and eval determinism')

Verified trainable params: 4,923,705
Output shape: (2, 10)
PASS: architecture forward, backward and eval determinism


## 3. Image Preprocessing, Augmentation, and Dataset Splitting

The dataset consists of 50,000 labeled CIFAR-10 training images and 2,000 competition test images with hidden labels.

Training transformations include random cropping with four-pixel padding, horizontal flipping, and the CIFAR-10 AutoAugment policy. These transformations introduce image variations to improve generalization.

Validation and test images receive only tensor conversion and normalization. No random augmentation is applied during ordinary evaluation.

Inputs are normalized using fixed RGB means `(0.4914, 0.4822, 0.4465)` and standard deviations `(0.2470, 0.2435, 0.2616)`.

The dataset-loading behavior depends on the selected execution mode:

- **Validate:** Stratified 45,000/5,000 training-validation split using seed 2026.
- **Full:** All 50,000 labeled images are used for training.
- **Predict:** Training-data loading is skipped because an existing full-data checkpoint is used.

The 2,000 test labels are not provided, so local test accuracy cannot be computed.


In [ ]:
# Fixed CIFAR-10 normalization values applied consistently
# during training, validation, and competition inference.
MEAN=(0.4914,0.4822,0.4465); STD=(0.2470,0.2435,0.2616)

# Optional Cutout augmentation masks a square image region.
# CUTOUT_SIZE is set to 0 in the final experiment, so this
# augmentation is defined but disabled.

class Cutout:
    def __init__(self,length): self.length=length
    def __call__(self,t):
        h,w=t.shape[-2:]; cy=random.randrange(h); cx=random.randrange(w)
        y1=max(0,cy-self.length//2); y2=min(h,cy+self.length//2)
        x1=max(0,cx-self.length//2); x2=min(w,cx+self.length//2)
        t=t.clone(); t[:,y1:y2,x1:x2]=0
        return t

# Construct separate transformations for training and evaluation.
# Training: random crop, horizontal flip, AutoAugment, normalization.
# Evaluation: tensor conversion and normalization only.

def build_transform(train):
    ops=([transforms.RandomCrop(32,padding=4),transforms.RandomHorizontalFlip(),
          transforms.AutoAugment(policy=AUTOAUGMENT_POLICY)] if train else [])
    ops += [transforms.ToTensor(),transforms.Normalize(MEAN,STD)]
    if train and CUTOUT_SIZE>0: ops.append(Cutout(CUTOUT_SIZE))
    return transforms.Compose(ops)

# Dataset wrapper that reads image filenames and integer labels
# from the competition CSV, loads RGB images, and applies the
# requested preprocessing transform.

class LocalCSVImages(Dataset):
    def __init__(self,frame,root,transform,path_col,label_col=None):
        self.frame=frame.reset_index(drop=True)
        self.root=Path(root); self.transform=transform
        self.path_col=path_col; self.label_col=label_col
    def __len__(self): return len(self.frame)
    def __getitem__(self,i):
        row=self.frame.iloc[i]; raw=str(row[self.path_col]); p=Path(raw)
        if not p.is_absolute():
            p=self.root / raw
            if not p.is_file():
                stem=Path(raw).stem
                choices=[self.root/(stem+ext) for ext in ('.png','.jpg','.jpeg','.bmp','.webp')]
                p=next((x for x in choices if x.is_file()),p)
        with Image.open(p) as img: x=self.transform(img.convert('RGB'))
        return (x,int(row[self.label_col])) if self.label_col is not None else x

def find_cifar_root(root):
    hits=list(root.rglob('cifar-10-batches-py')) if root.exists() else []
    return hits[0].parent if len(hits)==1 else None

def discover_training_data():
    root=Path(TRAIN_DIR) if TRAIN_DIR else None
    if TRAIN_CSV:
        frame=pd.read_csv(TRAIN_CSV)
        lcol=next((x for x in ('label','target','class') if x in frame.columns),None)
        pcol=next((x for x in ('filename','file','image','image_path','path','id') if x in frame.columns),None)
        if not lcol or not pcol: raise ValueError(f'Unexpected train CSV columns: {list(frame.columns)}')
        root=root or Path(TRAIN_CSV).parent
        assert root.is_dir(), f'Training image directory missing: {root}'
        assert not frame[pcol].isna().any() and not frame[lcol].isna().any()
        assert not frame[pcol].duplicated().any(), 'Duplicate training filenames'
        missing=[str(v) for v in frame[pcol] if not (root/str(v)).is_file()]
        if missing: raise FileNotFoundError(f'{len(missing)} training images missing; examples: {missing[:5]}')
        def factory(t): return LocalCSVImages(frame,root,t,pcol,lcol)
        targets=frame[lcol].astype(int).tolist()
        return factory,targets, f'CSV={TRAIN_CSV} images={root}'
    if root and root.is_dir() and any(p.is_dir() for p in root.iterdir()):
        tmp=ImageFolder(root)
        if len(tmp.classes)!=10: raise ValueError(f'Expected exactly 10 class dirs, found {tmp.class_to_idx}')
        print('ImageFolder class mapping:',tmp.class_to_idx)
        print('VERIFY that alphabetic class indices match evaluation integer indices before using this layout.')
        def factory(t): return ImageFolder(root,transform=t)
        return factory,tmp.targets,f'ImageFolder={root}'
    cifarroot=find_cifar_root(DATA_ROOT)
    if cifarroot is not None:
        base=CIFAR10(root=str(cifarroot),train=True,download=False)
        def factory(t): return CIFAR10(root=str(cifarroot),train=True,download=False,transform=t)
        return factory,base.targets,f'CIFAR10 python batch={cifarroot}'
    raise RuntimeError('No supported training layout detected. Inspect files and set TRAIN_DIR or TRAIN_CSV. Training will not guess or download data.')
# Choose the training dataset according to execution mode.
# Validate mode reserves 5,000 stratified images for evaluation.
# Full mode trains on all 50,000 labeled images.
# Predict mode skips training-data loading entirely.
#
# The validation indices are disjoint from the training indices,
# preventing training/validation image overlap.

if MODE in ('validate','full'):
    factory, targets, data_desc=discover_training_data()
    targets=np.asarray(targets,dtype=int)
    assert len(targets)==50_000, f'Expected 50000 authorized labeled images: got {len(targets)}. Confirm competition training set.'
    assert set(np.unique(targets))==set(range(10)), f'Labels must be 0..9: {np.unique(targets)}'
    print('Authorized training source:',data_desc,'class counts:',dict(Counter(targets.tolist())))
    if MODE=='validate':
        tr_idx,va_idx=train_test_split(np.arange(len(targets)),test_size=5000,random_state=SEED,stratify=targets)
        train_ds=Subset(factory(build_transform(True)),tr_idx)
        valid_ds=Subset(factory(build_transform(False)),va_idx)
        assert set(tr_idx).isdisjoint(set(va_idx))
        print('train:',len(train_ds),'validation:',len(valid_ds))
    else:
        train_ds=factory(build_transform(True));valid_ds=None
elif MODE=='predict':
    train_ds=None;valid_ds=None;data_desc='prediction-only'
else: raise ValueError('Invalid MODE')


## 4. Training Procedure: SGD, MixUp/CutMix, SAM, and EMA

This section implements the final optimization strategy.

### Optimization

We train for 240 epochs using SGD with momentum 0.9, Nesterov acceleration, weight decay 0.0005, and gradient-norm clipping at 10. The learning rate uses five warmup epochs followed by cosine decay.

### MixUp and CutMix

During the first 210 epochs, each batch is mixed with probability 0.65.

When mixing is selected:
- MixUp (50% conditional probability) linearly combines images and their labels using a Beta distribution with alpha 0.2.
- CutMix (50% conditional probability) replaces an image region with a patch from another training image and weights the labels according to the actual patch area.

Mixing is disabled during the final 30 epochs, allowing the model to finish training on intact augmented images.

### Sharpness-Aware Minimization (SAM)

SAM performs two forward/backward passes per batch. The first calculates a small weight perturbation along the gradient direction. The second evaluates the loss at the perturbed weights. The perturbation is removed before the optimizer update.

Both passes use the same augmented inputs and soft targets. Additional BatchNorm running-statistic updates are suppressed during the second pass.

SAM radius: 0.05.

### Exponential Moving Average (EMA)

Starting at epoch 10, the notebook maintains an EMA copy of the network weights using decay 0.999.

EMA smooths successive parameter updates and can stabilize predictions without increasing the parameter count of the final deployed classifier.

### Checkpoint selection

In validation mode, the best checkpoint is selected according to held-out accuracy, comparing raw and EMA predictions.

In full mode, the final epoch's EMA weights are saved because no validation split exists to select a checkpoint.

Checkpoints also store optimizer state, learning-rate scheduler state, training history, configuration, and random-number-generator states to support training resumption.


In [ ]:
def worker_init_fn(worker_id):
    s=torch.initial_seed()%2**32; np.random.seed(s); random.seed(s)

def loader(ds,shuffle):
    kwargs=dict(batch_size=BATCH_SIZE,shuffle=shuffle,num_workers=NUM_WORKERS,
                pin_memory=(DEVICE.type=='cuda'),worker_init_fn=worker_init_fn)
    if NUM_WORKERS>0: kwargs.update(persistent_workers=True,prefetch_factor=2)
    return DataLoader(ds,**kwargs)

train_loader=loader(train_ds,True) if train_ds is not None else None
val_loader=loader(valid_ds,False) if valid_ds is not None else None

# Configure Nesterov SGD, momentum, and weight decay.
# The optimizer updates the raw training model.

optimizer=torch.optim.SGD(model.parameters(),lr=LR,momentum=MOMENTUM,weight_decay=WEIGHT_DECAY,nesterov=True)

# Apply a five-epoch learning-rate warmup followed by
# cosine decay to a small final learning rate.
def lr_multiplier(epoch):
    if epoch < WARMUP_EPOCHS: return (epoch+1)/WARMUP_EPOCHS
    phase=(epoch-WARMUP_EPOCHS)/max(1,EPOCHS-WARMUP_EPOCHS-1)
    return 0.001+0.999*0.5*(1+math.cos(math.pi*min(1.,phase)))
scheduler=torch.optim.lr_scheduler.LambdaLR(optimizer,lr_lambda=lr_multiplier)
use_amp=(DEVICE.type=='cuda' and torch.cuda.is_bf16_supported())
amp_dtype=torch.bfloat16

# Create a separate EMA copy of the network.
# EMA parameters do not receive gradients directly;
# they are updated by averaging the raw model weights.

ema_model=copy.deepcopy(model).eval()
for p in ema_model.parameters(): p.requires_grad_(False)

# Record the architecture, hyperparameters, dataset mode,
# and augmentation settings for reproducibility.
# These settings are stored with the checkpoints.

run_config=dict(architecture='WRN28-ch16_59_117_235',params=PARAMS,seed=SEED,mode=MODE,source=data_desc,
    epochs=EPOCHS,batch_size=BATCH_SIZE,lr=LR,weight_decay=WEIGHT_DECAY,
    momentum=MOMENTUM,sam_rho=SAM_RHO,ema_decay=EMA_DECAY,ema_start=EMA_START_EPOCH,
    warmup_epochs=WARMUP_EPOCHS,augmentation="AutoAugment-MixUp-CutMix-last30AA-wrn28",
    mix_prob=MIX_PROB,mixup_alpha=MIXUP_ALPHA,cutmix_alpha=CUTMIX_ALPHA,
    mixup_fraction=MIXUP_FRACTION,final_unmixed_epochs=FINAL_UNMIXED_EPOCHS,
    cutout=CUTOUT_SIZE,
    label_smoothing=LABEL_SMOOTHING,train_count=len(train_ds) if train_ds is not None else 0)
run_id=f'{MODE}_wrn28_p{PARAMS}_sam{str(SAM_RHO).replace(".","p")}_aa_mixup_cutmix_p65_last30_ema{str(EMA_DECAY).replace(".","p")}_seed{SEED}'
last_path=OUTPUT/f'{run_id}_last.pt';best_path=OUTPUT/f'{run_id}_best.pt'
config_path=OUTPUT/f'{run_id}_config.json'
if MODE in ('validate','full'):
    if config_path.exists():
        previous=json.loads(config_path.read_text())
        if RESUME and last_path.exists():
            assert previous==run_config, 'Resume settings changed; restore config or use new experiment name.'
        elif not RESUME:
            raise RuntimeError('Existing run ID with RESUME=False. Change SEED to start a new run safely.')
    config_path.write_text(json.dumps(run_config,indent=2))
start_epoch=0; best_acc=-1.; best_kind=None; history=[]
if RESUME and MODE in ('validate','full') and last_path.is_file():
    ckpt=torch.load(last_path,map_location="cpu",weights_only=False)
    assert ckpt['config']==run_config, 'Resume configuration mismatch'
    model.load_state_dict(ckpt['raw_model']);ema_model.load_state_dict(ckpt['ema_model'])
    optimizer.load_state_dict(ckpt['optimizer']);scheduler.load_state_dict(ckpt['scheduler'])
    start_epoch=ckpt['epoch']+1;best_acc=ckpt['best_acc'];best_kind=ckpt['best_kind'];history=ckpt['history']
    torch.set_rng_state(ckpt['torch_rng'].cpu())
    np.random.set_state(ckpt['numpy_rng']);random.setstate(ckpt['python_rng'])
    if DEVICE.type=='cuda' and ckpt.get('cuda_rng') is not None: torch.cuda.set_rng_state_all(ckpt['cuda_rng'])
    print('Resumed SAM+EMA at epoch',start_epoch)

@torch.no_grad()
def ema_update():
    msd=model.state_dict(); esd=ema_model.state_dict()
    for key,old in esd.items():
        new=msd[key].detach()
        if old.is_floating_point(): old.lerp_(new,1.-EMA_DECAY)
        else: old.copy_(new)

def batchnorm_momentum_off(net):
    states=[]
    for mod in net.modules():
        if isinstance(mod,nn.modules.batchnorm._BatchNorm):
            states.append((mod,mod.momentum,mod.num_batches_tracked.clone() if mod.num_batches_tracked is not None else None))
            mod.momentum=0.0
    return states

def batchnorm_momentum_restore(states):
    for mod,momentum,counter in states:
        mod.momentum=momentum
        if counter is not None: mod.num_batches_tracked.copy_(counter)

# Apply MixUp, CutMix, or no sample mixing.
# Produce soft class targets, accounting for the mixing
# coefficient or the actual area of the CutMix patch.
# The same mixed images and targets are reused for both
# passes of SAM.

def mix_batch(x, y, epoch):
    """Create ONE mixed tensor and ONE soft target, reused for both SAM passes."""
    target=F.one_hot(y,num_classes=10).to(dtype=torch.float32)
    if epoch >= EPOCHS-FINAL_UNMIXED_EPOCHS or random.random() >= MIX_PROB:
        return x,target,'none'
    perm=torch.randperm(x.shape[0],device=x.device)
    if random.random() < MIXUP_FRACTION:
        lam=float(np.random.beta(MIXUP_ALPHA,MIXUP_ALPHA))
        return x*lam+x[perm]*(1-lam),target*lam+target[perm]*(1-lam),'mixup'
    lam=float(np.random.beta(CUTMIX_ALPHA,CUTMIX_ALPHA))
    h,w=x.shape[-2:]
    cut_ratio=math.sqrt(1-lam)
    ch=int(round(h*cut_ratio));cw=int(round(w*cut_ratio))
    cy=random.randrange(h);cx=random.randrange(w)
    y1=max(0,cy-ch//2);y2=min(h,cy+(ch+1)//2)
    x1=max(0,cx-cw//2);x2=min(w,cx+(cw+1)//2)
    result=x.clone()
    result[:,:,y1:y2,x1:x2]=x[perm,:,y1:y2,x1:x2]
    actual_lam=1-((y2-y1)*(x2-x1))/(h*w)
    return result,target*actual_lam+target[perm]*(1-actual_lam),'cutmix'

# Compute cross-entropy against soft targets produced by
# MixUp/CutMix, incorporating label smoothing of 0.05.

def mixed_ce(logits,soft_target):

    soft_target=(1-LABEL_SMOOTHING)*soft_target+LABEL_SMOOTHING/10
    return -(soft_target * F.log_softmax(logits.float(),dim=1)).sum(1).mean()

# Train one epoch using Sharpness-Aware Minimization:
# 1. Calculate loss and gradients on the current weights.
# 2. Perturb weights in the normalized gradient direction.
# 3. Recalculate loss and gradients at perturbed weights.
# 4. Restore the original weights.
# 5. Clip gradients and execute the SGD optimizer step.
# 6. Update the EMA model when the EMA start epoch is reached.
#
# Training accuracy is measured only on unmixed batches,
# since mixed-image targets do not have one unambiguous class.

def sam_train_epoch():
    model.train(); correct=total=0;loss_sum=0.0;mix_counts=Counter();unmixed_examples=0
    for x,y in train_loader:
        x=x.to(DEVICE,non_blocking=True);y=y.to(DEVICE,non_blocking=True)
        x,soft_y,mix_kind=mix_batch(x,y,current_epoch)
        mix_counts[mix_kind]+=1
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=DEVICE.type,dtype=amp_dtype,enabled=use_amp):
            logits=model(x)
            loss=mixed_ce(logits,soft_y)
        loss.backward()
        # Smoke test: verify that two 32x32 RGB inputs produce
        # an output tensor of shape (2, 10), one logit per class.
        with torch.no_grad():
            grad_norm=torch.linalg.vector_norm(torch.stack([
                p.grad.detach().float().norm(p=2) for p in model.parameters() if p.grad is not None
            ]),ord=2)
            if not torch.isfinite(grad_norm): raise FloatingPointError('Non-finite SAM gradient norm')
            scale=SAM_RHO/(grad_norm.item()+1e-12)
            perturb=[]
            for p in model.parameters():
                if p.grad is not None:
                    e=p.grad.detach()*scale
                    p.add_(e);perturb.append((p,e))
        optimizer.zero_grad(set_to_none=True)
        bn_states=batchnorm_momentum_off(model)
        try:
            with torch.autocast(device_type=DEVICE.type,dtype=amp_dtype,enabled=use_amp):
                loss_perturbed=mixed_ce(model(x),soft_y)
            loss_perturbed.backward()
        finally:
            batchnorm_momentum_restore(bn_states)
            with torch.no_grad():
                for p,e in perturb: p.sub_(e)
        torch.nn.utils.clip_grad_norm_(model.parameters(),max_norm=10.0,error_if_nonfinite=True)
        optimizer.step()
        if current_epoch+1>=EMA_START_EPOCH: ema_update()
        b=y.size(0);total+=b;loss_sum+=float(loss.detach())*b
        if mix_kind=='none':
            correct+=(logits.argmax(1)==y).sum().item()
            unmixed_examples+=b

    unmixed_total=unmixed_examples
    return loss_sum/total,mix_counts,correct,unmixed_total

@torch.inference_mode()
def evaluate(which_model):
    which_model.eval(); correct=total=0;loss_sum=0.0
    for x,y in val_loader:
        x=x.to(DEVICE,non_blocking=True);y=y.to(DEVICE,non_blocking=True)
        with torch.autocast(device_type=DEVICE.type,dtype=amp_dtype,enabled=use_amp):
            logits=which_model(x).float()
        correct+=(logits.argmax(1)==y).sum().item()
        total+=y.size(0);loss_sum+=F.cross_entropy(logits,y,reduction='sum').item()
    return loss_sum/total, correct/total

def save_checkpoint(path,epoch,prediction_model='raw'):
    selected=ema_model if prediction_model=='ema' else model
    ckpt=dict(epoch=epoch,model=selected.state_dict(),raw_model=model.state_dict(),
        ema_model=ema_model.state_dict(),prediction_model=prediction_model,
        optimizer=optimizer.state_dict(),scheduler=scheduler.state_dict(),
        best_acc=best_acc,best_kind=best_kind,history=history,config=run_config,
        torch_rng=torch.get_rng_state(),numpy_rng=np.random.get_state(),python_rng=random.getstate(),
        cuda_rng=torch.cuda.get_rng_state_all() if DEVICE.type=='cuda' else None)
    temp=Path(str(path)+'.tmp');torch.save(ckpt,temp);temp.replace(path)

# Execute the training loop only in validate or full mode.
#
# Validate mode:
# Evaluate raw and EMA models after each epoch and save
# the checkpoint with the highest held-out accuracy.
#
# Full mode:
# Train on the entire labeled dataset without validation
# and save the final EMA model at epoch 240.
#
# Save a latest checkpoint and CSV training history after
# every epoch to permit recovery and result inspection.

if MODE in ('validate','full'):
    for current_epoch in range(start_epoch,EPOCHS):
        t0=time.perf_counter()
        train_loss,mix_counts,correct_unmixed,total_unmixed=sam_train_epoch()
        train_acc=(correct_unmixed/total_unmixed) if total_unmixed else float("nan")
        raw_val=ema_val=float('nan')
        if val_loader is not None:
            _,raw_val=evaluate(model)
            if current_epoch+1>=EMA_START_EPOCH:
                _,ema_val=evaluate(ema_model)
            if raw_val>best_acc:
                best_acc=raw_val;best_kind='raw';save_best=True
            else: save_best=False
            if not math.isnan(ema_val) and ema_val>best_acc:
                best_acc=ema_val;best_kind='ema';save_best=True
        else:
            save_best=(current_epoch+1==EPOCHS)
            best_kind='ema' if current_epoch+1>=EMA_START_EPOCH else 'raw'
        record=dict(epoch=current_epoch+1,train_loss=train_loss,train_acc=train_acc,
            raw_val=raw_val,ema_val=ema_val,best_val=best_acc,best_kind=best_kind,
            mixup_batches=mix_counts["mixup"],cutmix_batches=mix_counts["cutmix"],plain_batches=mix_counts["none"],
            lr=optimizer.param_groups[0]['lr'],seconds=time.perf_counter()-t0)
        history.append(record)

        scheduler.step()
        save_checkpoint(last_path,current_epoch,prediction_model=best_kind or 'raw')
        if save_best: save_checkpoint(best_path,current_epoch,prediction_model=best_kind)
        tmp=OUTPUT/f'{run_id}_history.csv.tmp'
        pd.DataFrame(history).to_csv(tmp,index=False)
        tmp.replace(OUTPUT/f'{run_id}_history.csv')
        print(f'epoch {current_epoch+1:3d}/{EPOCHS} train={train_acc:.2%} '
              f'raw_val={raw_val:.2%} ema_val={ema_val:.2%} best={best_acc:.2%} '
              f'best_type={best_kind} mixup={mix_counts["mixup"]} cutmix={mix_counts["cutmix"]} plain={mix_counts["none"]} seconds={record["seconds"]:.1f}',flush=True)
    print('FULL/VALIDATE COMPLETE. Selected checkpoint:',best_path)
    if val_loader is not None: print('Best validation accuracy:',f'{best_acc:.2%}','using',best_kind)


## 5. Validation Analysis and Robustness Checks

This section runs only in `validate` mode after training.

The notebook loads the checkpoint selected by validation accuracy and reports the top five epochs from the training history.

It then performs simple robustness checks on the held-out validation set by applying small brightness and contrast changes. These tests examine whether the selected classifier maintains its accuracy under modest image appearance variations.

The checks are:
- Original validation images.
- Brightness increased by 8%.
- Contrast increased by 8%.
- Brightness and contrast both increased by 8%.

The results are written to a CSV for analysis.

**Final development result:** The selected WRN-28 EMA checkpoint achieved 96.98% accuracy on the 5,000-image validation set. This is separate from the final full-data model used for Kaggle submission.


In [ ]:
if MODE=='validate' and best_path.exists():
    df=pd.read_csv(OUTPUT/f'{run_id}_history.csv')
    print('Top 5 epochs by best validation accuracy:')
    display(df.sort_values('best_val',ascending=False).head(5)[['epoch','raw_val','ema_val','best_val','best_kind']])
    print('Checkpoint to compare:',best_path)


    best_ckpt=torch.load(best_path,map_location=DEVICE,weights_only=False)
    selected_model=ema_model if best_ckpt['prediction_model']=='ema' else model
    selected_model.load_state_dict(best_ckpt['model'])
    selected_model.eval()
    from torchvision.transforms import functional as TF

    class ShiftedValidation(Dataset):
        def __init__(self,base,brightness=1.0,contrast=1.0):
            self.base=base;self.brightness=brightness;self.contrast=contrast
        def __len__(self): return len(self.base)
        def __getitem__(self,i):

            x,y=self.base[i]
            x=TF.normalize(x,mean=[-m/s for m,s in zip(MEAN,STD)],std=[1/s for s in STD])
            x=x.clamp(0,1)
            x=TF.adjust_brightness(x,self.brightness)
            x=TF.adjust_contrast(x,self.contrast)
            x=TF.normalize(x,MEAN,STD)
            return x,y

    @torch.inference_mode()
    def shifted_accuracy(brightness=1.0,contrast=1.0):
        dl=loader(ShiftedValidation(valid_ds,brightness,contrast),False)
        correct=total=0
        for x,y in dl:
            x=x.to(DEVICE,non_blocking=True);y=y.to(DEVICE,non_blocking=True)
            with torch.autocast(device_type=DEVICE.type,dtype=amp_dtype,enabled=use_amp):
                logits=selected_model(x).float()
            correct+=(logits.argmax(1)==y).sum().item();total+=y.size(0)
        return 100*correct/total

    checks={'clean':(1.0,1.0),'brightness +8%':(1.08,1.0),
            'contrast +8%':(1.0,1.08),'brightness +8%, contrast +8%':(1.08,1.08)}
    results={name:shifted_accuracy(*factors) for name,factors in checks.items()}
    print('Selected checkpoint:',best_ckpt['prediction_model'])
    print('Validation robustness (%):',results)
    pd.DataFrame([{'condition':name,'accuracy_pct':score} for name,score in results.items()]).to_csv(
        OUTPUT/f'{run_id}_robustness.csv',index=False)
else:
    print('Validation summary not applicable in',MODE)

Validation summary not applicable in predict


## 6. Final Model Inference and Kaggle Submission

This section runs in `predict` mode and generates the competition submission.

The model is not trained in this stage. Instead, the previously trained full-data WRN-28 checkpoint is loaded, and its configuration is verified.

### Test dataset

The competition provides 2,000 test images without their true class labels.

The code uses `test.csv` to associate image IDs with filenames and `sample_submission.csv` to establish the required output order.

### Test-time augmentation

When `USE_TTA=True`, each image is evaluated twice:

1. In its original orientation.
2. After horizontal flipping.

The two sets of class logits are averaged, and the highest-scoring class is selected as the predicted label.

This uses **the same trained model for both views**, not an ensemble of independently trained models.

### Submission output

The notebook generates a CSV with two columns:

`id,label`

Each `label` is an integer from 0 through 9. Assertions verify the output shape, class range, and submission ID ordering before saving.

The CSV is uploaded to Kaggle because the true test labels are hidden. Kaggle evaluates the predictions and reports the public leaderboard accuracy.

**Final Kaggle result: 88.80%.**


In [ ]:
def find_sample_csv():
    if SAMPLE_CSV: return Path(SAMPLE_CSV)
    matches=[p for p in DATA_ROOT.rglob('*.csv') if 'sample' in p.name.lower() and 'submission' in p.name.lower()]
    if len(matches)!=1: raise RuntimeError(f'Expected unique sample submission; found {matches}. Set SAMPLE_CSV.')
    return matches[0]

def find_test_dir():
    if TEST_DIR: return Path(TEST_DIR)
    dirs=[p for p in DATA_ROOT.rglob('*') if p.is_dir() and p.name.lower() in ('test','test_images','test-images')]
    if len(dirs)!=1: raise RuntimeError(f'Ambiguous test directory: {dirs}. Set TEST_DIR.')
    return dirs[0]

class SubmissionImages(Dataset):
    def __init__(self,ids,folder,transform):
        self.ids=[str(x) for x in ids];self.transform=transform
        paths=[p for p in folder.rglob('*') if p.is_file() and p.suffix.lower() in ('.png','.jpg','.jpeg','.bmp','.webp')]
        by_stem={}
        for p in paths:
            if p.stem in by_stem: raise RuntimeError(f'Duplicate test filename stem: {p.stem}')
            by_stem[p.stem]=p
        self.paths=[]
        for id_ in self.ids:
            path=by_stem.get(Path(id_).stem)
            if path is None: raise FileNotFoundError(f'No image found for id={id_!r} in {folder}')
            self.paths.append(path)
        if len(set(self.paths))!=len(self.paths): raise RuntimeError('Multiple rows mapped to same test image')
    def __len__(self): return len(self.ids)
    def __getitem__(self,i):
        with Image.open(self.paths[i]) as img: return self.transform(img.convert('RGB'))

if MODE=='predict':
    sample_path=find_sample_csv();sample=pd.read_csv(sample_path,dtype={'id':str})
    if list(sample.columns)!=['id','label']: raise ValueError(f'Expected columns id,label; found {list(sample.columns)}')
    if sample.empty or sample['id'].isna().any() or sample['id'].duplicated().any(): raise ValueError('Invalid sample submission IDs')
    folder=find_test_dir()
    test_index=DATA_ROOT/'test.csv'
    if not test_index.is_file():
        raise FileNotFoundError(f'Missing test.csv mapping: {test_index}')
    test_frame=pd.read_csv(test_index,dtype={'id':str,'filename':str})
    if not {'id','filename'}.issubset(test_frame.columns):
        raise ValueError(f'Unexpected test.csv columns: {list(test_frame.columns)}')
    if test_frame['id'].isna().any() or test_frame['filename'].isna().any():
        raise ValueError('Missing test IDs or filenames')
    if test_frame['id'].duplicated().any() or test_frame['filename'].duplicated().any():
        raise ValueError('Duplicate test IDs or filenames')
    mapping=dict(zip(test_frame['id'],test_frame['filename']))
    if set(mapping)!=set(sample['id']):
        raise ValueError('test.csv IDs differ from sample_submission.csv IDs')
    filenames=[mapping[id_] for id_ in sample['id']]
    test_data=SubmissionImages(filenames,folder,build_transform(False))
    test_loader=loader(test_data,False)
    chosen=Path(CHECKPOINT_PATH) if CHECKPOINT_PATH else None
    if chosen is None or not chosen.exists():

        raise FileNotFoundError('Set CHECKPOINT_PATH to a trained validation/full checkpoint; no model was trained in prediction mode.')
    ckpt=torch.load(chosen,map_location='cpu',weights_only=False)
    if (ckpt.get('config',{}).get('mode')!='full' or
        ckpt.get('config',{}).get('sam_rho')!=SAM_RHO or
        ckpt.get('config',{}).get('params')!=PARAMS or
        ckpt.get('config',{}).get('augmentation')!='AutoAugment-MixUp-CutMix-last30AA-wrn28'):
        raise ValueError('Prediction requires the matching NEW full-data SAM checkpoint, not a validation/baseline checkpoint')
    print('Loaded trained checkpoint type:',ckpt.get('prediction_model'),'epoch:',ckpt.get('epoch',-1)+1)
    model.load_state_dict(ckpt['model'],strict=True);model.eval()
    labels=[]
    with torch.inference_mode():
        for x in test_loader:
            x=x.to(DEVICE,non_blocking=True)
            with torch.autocast(device_type=DEVICE.type,enabled=use_amp):
                logits=model(x).float()
                if USE_TTA: logits=(logits+model(torch.flip(x,[-1])).float())/2
            labels.extend(logits.argmax(1).cpu().tolist())
    assert len(labels)==len(sample)
    submission=pd.DataFrame({'id':sample['id'].tolist(),'label':labels})
    assert submission['id'].tolist()==sample['id'].tolist()
    assert submission['label'].notna().all() and submission['label'].between(0,9).all()
    assert all(type(x)==int for x in labels)

    timestamp = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')

    out = DRIVE_PROJECT / (
        f'submission_wrn28_p4923705_full_ema_{timestamp}.csv'
    )

    temp_out = out.with_suffix('.csv.tmp')
    submission.to_csv(temp_out, index=False)
    temp_out.replace(out)
    verify=pd.read_csv(out,dtype={'id':str})
    assert list(verify.columns)==['id','label'] and verify['id'].tolist()==sample['id'].tolist()
    assert verify['label'].between(0,9).all() and len(verify)==len(sample)
    print('Saved verified submission:',out,'rows:',len(verify),'checkpoint:',chosen)
else: print('Prediction cell skipped; set MODE=predict after training.')

Loaded trained checkpoint type: ema epoch: 240
Saved verified submission: /content/drive/MyDrive/CIFAR10_Competition/submission_wrn28_p4923705_full_ema_20261106_145851_957000.csv rows: 2000 checkpoint: /content/drive/MyDrive/CIFAR10_Competition/runs/full_wrn28_p4923705_sam0p05_aa_mixup_cutmix_p65_last30_ema0p999_seed2026_best.pt


## 7. Results and Key Findings

### Final model

| Metric | Result |
|---|---|
| Architecture | Custom preactivation Wide ResNet-28 |
| Trainable parameters | 4,923,705 |
| Parameter limit | 5,000,000 |
| Development training set | 45,000 images |
| Development validation set | 5,000 images |
| Final training set | 50,000 images |
| Competition test set | 2,000 unlabeled images |
| Training epochs | 240 |
| Best local validation accuracy | 96.98% |
| Kaggle public leaderboard accuracy | 88.80% |

### Findings

The final architecture uses residual learning, increased channel width, and regularization techniques while remaining below the competition parameter limit.

AutoAugment, MixUp/CutMix, SAM, and EMA were combined into the selected training recipe. These components were not all individually ablated in the final WRN-28 configuration, so their separate contributions cannot be established from this experiment alone.

Although the validation checkpoint achieved 96.98% accuracy, the full-data model received 88.80% on the Kaggle public leaderboard. The difference highlights the limitations of relying on local validation accuracy alone when assessing performance on an independently evaluated hidden-label test set.

All competition predictions were generated using one trained residual network, with horizontal-flip test-time augmentation. The model was trained using only the authorized labeled data, without external pretrained weights.

The final submission demonstrates that a carefully regularized Wide ResNet can achieve strong image classification accuracy within a strict five-million-parameter budget.
